In [ ]:
from eregion.tasks import ImageCreator, ImageBundle, ImageResult
from eregion.tasks.custom import guess_image_type_from_filename_DEIMOS, load_image_fits_DEIMOS
from eregion.tasks.calibration import MasterBias, CalibrationResult
from eregion.tasks.preprocessing import BiasSubtraction, ScanSubtraction, SigmaClipMasking

import numpy as np
import matplotlib.pyplot as plt
import os
import glob2
from copy import deepcopy
import importlib

from tasks.ptc import PTCResult

# Example: DEIMOS science focal plane characterization

In [ ]:
basepath = '/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/PTC/SCI'
runid = '20260718*'
rawpath = glob2.glob(os.path.join(basepath, runid))[0]
outpath = rawpath.replace('/DTU_dettest/','/DTU_detreduce/')
if not os.path.exists(outpath):
    os.makedirs(outpath)

### Calproc

In [ ]:
# Load bias images
creator = ImageCreator(detector_config = '../eregion/configs/detectors/deimos_sci.yaml')
res = creator.run(input_source = os.path.join(rawpath,'*bias*.fits'),
                  identifier_func = guess_image_type_from_filename_DEIMOS,
                  fileloader_func = load_image_fits_DEIMOS,
                  data_on_demand = True)

# make master bias
master_bias_task = MasterBias(method='median')
mb_res = master_bias_task.run(bias_images=res.data('type == "bias"'))


In [ ]:
# cleanup memory
del res
# save master bias result obj
mb_res.save(outpath)

In [ ]:
# load master bias result obj
mb_res = CalibrationResult.load(outpath)
mb_res.master_bias


### Preproc

Sanity checks for each step

In [ ]:
def quick_plot(img):
    vmin, vmax = np.percentile(img.data.values, [5, 95])
    img.show(cmap='gray', vmin=vmin, vmax=vmax)

In [ ]:
## Loadig first pair at exptime 0
creator = ImageCreator(detector_config='../eregion/configs/detectors/deimos_sci.yaml', max_batch_size=2)

flpair = creator.run(input_source='/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/PTC/SCI/20260718*/*SCI_1_flat_0.000*.fits',
                  identifier_func=guess_image_type_from_filename_DEIMOS,
                  fileloader_func=load_image_fits_DEIMOS,
                  data_on_demand=True)
quick_plot(flpair.data[0])

In [ ]:
# Do bias sub
bias_sub = BiasSubtraction(quick=True)

after_bs = bias_sub.run(images=deepcopy(flpair.data), master_bias=mb_res.master_bias)
quick_plot(after_bs.data[0])

In [ ]:
# Do overscan sub
oscan_sub = ScanSubtraction(which_scan='serial_overscan', method='median_by_axis')

after_osub = oscan_sub.run(images=deepcopy(after_bs.data))
quick_plot(after_osub.data[0])

In [ ]:
# Do sigma clip
cr_mask = SigmaClipMasking(sigma_clip_args={'sigma_lower':5, 'sigma_upper':5}, n_jobs=1) # set n_jobs=1 to be able to debug in pycharm inside parallelized jobs

after_cr = cr_mask.run(images=deepcopy(after_osub.data))
quick_plot(after_cr.data[0])

In [ ]:
# Do PTC
import eregion.tasks.ptc as ptc
importlib.reload(ptc)

ptc_task = ptc.PTC(n_jobs=1)
pres = ptc_task.run(images=deepcopy(after_cr.data))

In [ ]:
pres.ptc_table

In [ ]:
# save to fits
pres.save(outpath)

In [ ]:
# load from fits
pres = ptc.PTCResult.load(outpath)
pres.ptc_table

### Everything together

In [ ]:
creator = ImageCreator(detector_config='../eregion/configs/detectors/deimos_sci.yaml', max_batch_size=2)
bias_sub = BiasSubtraction()
oscan_sub = ScanSubtraction(which_scan='serial_overscan', method='median_by_axis')
cr_mask = SigmaClipMasking(sigma_clip_args={'sigma_lower':5, 'sigma_upper':5})
ptc_task = ptc.PTC()

count = 0
# Load flat images in batches (lazy mode)
for flpair in creator.lazy_run(input_source='/Users/yashvi/Desktop/Detector Characterization Tools/DTU_dettest/DTU_fullfp_bringup/PTC/SCI/20260718*/*flat*.fits',
                  identifier_func=guess_image_type_from_filename_DEIMOS,
                  fileloader_func=load_image_fits_DEIMOS,
                  data_on_demand=True):
    flpair = bias_sub.run(images=flpair.data, master_bias=mb_res.master_bias)
    flpair = oscan_sub.run(images=flpair.data)
    flpair = cr_mask.run(images=flpair.data)
    ptcres = ptc_task.run(images=flpair.data)

    if count==0:
        preproc_res = flpair
        ptc_res = ptcres
    else:
        preproc_res = preproc_res.combine(flpair)
        ptc_res = ptc_res.combine(ptcres)
    count+=1
    print('Pairs processed: ', count)
    if count>=1:
        break


In [ ]:
ptc_res.save(outpath)